In [1]:
import random

import torch
import torchvision
import torchvision.transforms as transforms

In [2]:
from models import ResNet18
import utils

/vol/home/s4015355/.local/lib/python3.10/site-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# model = ResNet18()

In [4]:
# def ResNet18(in_shape=(1, 32, 32), num_classes=10, base_channels=64):
#     block_nums = [2, 2, 2, 2]
#     block_type = "basic"
#     return ResNet(in_shape, num_classes, base_channels, block_nums, block_type)

path = "save/ckpts/trainings_resnet/B7A869.pt"
pt_model_state_dict = torch.load(path)['B7A86935']['best_state']
pt_model = ResNet18(in_shape=(3,32,32,), num_classes=10)
pt_model.load_state_dict(pt_model_state_dict)
ex_input = torch.rand(1,3,32,32)
onnx_model = torch.onnx.export(pt_model, ex_input, "attias.onnx")

============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [5]:
path = "save/ckpts/trainings_resnet/B7A869.pt"
info = torch.load(path)['B7A86935']
pt_model_state_dict = info['best_state']
pt_model = ResNet18(in_shape=(3,32,32,), num_classes=10, base_channels=64)
_ = pt_model.load_state_dict(pt_model_state_dict)
pt_model.eval()
print(_)

<All keys matched successfully>


In [6]:
def create_resnet_loaders(
    data_dir, task_config, train_config, num_workers=1, adversarial_augemntation=False
):

    if task_config["task"] == "CIFAR10":
        dataset = torchvision.datasets.CIFAR10(data_dir, download=True)
    
    sample_num = len(dataset)
    print(sample_num)
    valid_num = int(len(dataset) * train_config["valid_portion"])
    # valid_idxs = random.sample(range(sample_num), valid_num)
    valid_idxs = range(0,1000)
    train_idxs = list(set(range(sample_num)) - set(valid_idxs))

    if task_config["task"] == "CIFAR10":
        dataset_train = torch.utils.data.Subset(
            torchvision.datasets.CIFAR10(data_dir, train=True, transform= transforms.ToTensor()),
            train_idxs,
        )
        dataset_valid = torch.utils.data.Subset(
            torchvision.datasets.CIFAR10(data_dir, train=True, transform= transforms.ToTensor()),
            valid_idxs,
        )
        dataset_test = torchvision.datasets.CIFAR10(
            data_dir, train=False, transform= transforms.ToTensor()
        )

    batch_size = train_config["resnet_batch_size"]
    loader_train = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    loader_valid = torch.utils.data.DataLoader(
        dataset_valid, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    loader_test = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    return loader_train, loader_valid, loader_test

In [7]:
losses = {"train": [], "valid": [], "test": []}
accs = {"train": [], "valid": [], "test": []}
run_config = {'device': "cuda"}
utils.evaluate_all(
    pt_model, 
    run_config, 
    info["task_config"], 
    # utils.create_resnet_loaders('data', info['task_config'], info['train_config']),
    create_resnet_loaders('data', info['task_config'], info['train_config']),
    ["train", "valid", "test"],
    losses,
    accs)

Files already downloaded and verified
50000
evaluating ...
train dataset
cross entropy loss: 2.8731, accuracy:  33.20%
valid dataset
cross entropy loss: 2.9816, accuracy:  31.50%
test dataset
cross entropy loss: 2.8791, accuracy:  33.27%
